In [1]:
# Gerekli kütüphaneleri içe aktarır.

import random
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# Sonuçların tekrar üretilebilir olması için rastgelelik değerlerini sabitler.

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [3]:
# data.csv dosyasını okur ve ilk satırları gösterir.

df = pd.read_csv("data.csv")

print(df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data.csv'

In [ ]:
# Özellikleri X, hedef değişkeni y olarak ayırır.

X = df.drop("target", axis=1)
y = df["target"]

print("X:", X.shape)
print("y:", y.shape)

In [ ]:
# Veriyi train, validation ve test olarak ayrı ayrı ve sınıf oranlarını koruyarak böler.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

In [ ]:
# StandardScaler'ı yalnızca training verisinde öğrenir ve validation/test verisine uygular.

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
# Küçük tabular veri için dropout içeren, dengeli kapasiteli binary classification ağı oluşturur.

model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),

    layers.Dense(32, activation="relu"),
    layers.Dropout(0.15),

    layers.Dense(16, activation="relu"),
    layers.Dropout(0.10),

    layers.Dense(1, activation="sigmoid")
])

model.summary()

In [ ]:
# Modelin optimizer, loss fonksiyonu ve başarı metriğini belirler.

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy")
    ]
)

In [ ]:
# Validation loss düzelmediğinde öğrenme hızını azaltır ve gerekirse eğitimi erken durdurur.

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=4,
    min_lr=0.00001
)

In [ ]:
# Modeli ayrı validation setiyle eğitir ve en iyi ağırlıkları korur.

history = model.fit(
    X_train,
    y_train,

    validation_data=(X_val, y_val),

    epochs=10,
    batch_size=16,

    callbacks=[
        early_stopping,
        reduce_lr
    ],

    verbose=1
)

In [ ]:
# Eğitilmiş modeli test verisi üzerinde değerlendirir.

test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test Loss:", round(test_loss, 4))
print("Test Accuracy:", round(test_accuracy, 4))

In [ ]:
# Eğitim ve validation loss değerlerinin epoch boyunca değişimini çizer.

import matplotlib.pyplot as plt

plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
# Eğitim ve validation accuracy değerlerinin epoch boyunca değişimini çizer.

plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
# Test verisi için olasılık tahminlerini üretir ve 0.5 eşik değeriyle sınıfa çevirir.

y_prob = model.predict(X_test)

y_pred = (y_prob >= 0.5).astype(int).ravel()

print(y_pred[:10])

In [ ]:
# Confusion matrix ile doğru ve yanlış sınıflandırmaları gösterir.

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

In [ ]:
# Precision, recall, F1-score ve accuracy değerlerini ayrıntılı olarak gösterir.

from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Malignant", "Benign"]
    )
)

In [ ]:
# ROC-AUC skorunu hesaplayarak modelin sınıfları ayırma gücünü ölçer.

from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test, y_prob.ravel())

print("ROC-AUC:", round(auc, 4))